# docparse-vlm: zero-shot vs. LoRA-tuned Qwen2-VL-2B on CORD v2 receipts

Code: [github.com/katyazubareva/docparse-vlm](https://github.com/katyazubareva/docparse-vlm)

This notebook:
1. installs the package from GitHub,
2. evaluates **Qwen2-VL-2B-Instruct zero-shot** on the CORD v2 test split,
3. **LoRA fine-tunes** it on the 800 training receipts,
4. evaluates the adapter with the same metrics and compares the two,
5. shows the worst remaining errors.

**Settings:** Accelerator = *GPU T4 x2* (one GPU is enough), Internet = *On*.
Set `SMOKE = True` for a ~10 minute end-to-end check before the full run (about 1.5–2 h on a T4).

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
!pip install -q "docparse-vlm[train] @ git+https://github.com/katyazubareva/docparse-vlm.git"

In [ ]:
import os

SMOKE = False  # True: tiny run to check the pipeline end to end

OUT = "/kaggle/working/outputs"
ADAPTER = "/kaggle/working/runs/lora-r16"
EVAL_LIMIT = "--limit 10" if SMOKE else ""
TRAIN_ARGS = "--train-docs 32 --epochs 1" if SMOKE else "--epochs 3"

os.environ["CUDA_VISIBLE_DEVICES"] = "0"  # 2B in fp16 fits on one T4; avoids naive model parallelism
os.environ["TOKENIZERS_PARALLELISM"] = "false"

# Optional: add HF_TOKEN in Add-ons -> Secrets for faster, rate-limit-free Hub downloads.
try:
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
except Exception:
    pass

## 1. Metric sanity check
`oracle` must score F1 = 1 and `empty` must score F1 = 0. If either fails, the metric is broken, not the model.

In [ ]:
!python -m docparse.evaluate --system oracle --split test --out {OUT} | grep '"f1"'
!python -m docparse.evaluate --system empty  --split test --out {OUT} | grep '"f1"'

## 2. Zero-shot baseline (B2)

In [ ]:
!python -m docparse.evaluate --system qwen2vl-zeroshot --split test --out {OUT} {EVAL_LIMIT}

## 3. LoRA fine-tune (M1)
LoRA r=16 on the language model's attention + MLP projections; the vision tower is frozen. fp16 base weights, fp32 adapters, gradient checkpointing.

In [ ]:
!python -m docparse.train --output {ADAPTER} {TRAIN_ARGS}

## 4. Evaluate the adapter

In [ ]:
!python -m docparse.evaluate --system qwen2vl-lora --adapter {ADAPTER} --split test --out {OUT} {EVAL_LIMIT}

## 5. Comparison

In [ ]:
import json
import pandas as pd

rows = {}
for name in ["qwen2vl-zeroshot", "qwen2vl-lora"]:
    m = json.load(open(f"{OUT}/{name}-test/metrics.json"))
    lo, hi = m["f1_ci95"]
    rows[name] = {
        "n": m["n_docs"],
        "field F1": f'{m["f1"]:.3f} [{lo:.3f}, {hi:.3f}]',
        "precision": round(m["precision"], 3),
        "recall": round(m["recall"], 3),
        "doc exact": round(m["doc_exact"], 3),
        "JSON valid": round(m["json_valid"], 3),
        "total acc": round(m["total_acc"], 3),
        "p50 s": round(m["latency_p50_s"], 2),
        "p95 s": round(m["latency_p95_s"], 2),
    }
pd.DataFrame(rows).T

## 6. Error analysis: worst documents after fine-tuning

In [ ]:
from docparse.data import iter_split

preds = pd.read_json(f"{OUT}/qwen2vl-lora-test/predictions.jsonl", lines=True)
preds["missed"] = preds["n_gold"] - preds["tp"]
preds["spurious"] = preds["n_pred"] - preds["tp"]
worst = preds.sort_values(["missed", "spurious"], ascending=False).head(5)
print(f"invalid JSON: {(~preds['valid']).sum()} / {len(preds)}")
worst[["id", "tp", "n_gold", "n_pred", "missed", "spurious", "valid"]]

In [ ]:
import matplotlib.pyplot as plt

gold = {doc_id: (img, r) for doc_id, img, r in iter_split("test")}
for _, row in worst.head(3).iterrows():
    img, r = gold[row["id"]]
    plt.figure(figsize=(5, 7)); plt.imshow(img); plt.axis("off"); plt.title(row["id"]); plt.show()
    print("GOLD:", r.model_dump_json(exclude_none=True))
    print("PRED:", row["raw"][:1500], "\n")

## 7. Outputs
The adapter (`runs/lora-r16`) and all predictions/metrics (`outputs/`) are saved in `/kaggle/working`, so they appear under this notebook's **Output** tab and can be attached to other notebooks as a dataset.